# 🚘 Vehicle License Plate Detection using Detectron2

This notebook builds a computer vision pipeline for detecting vehicle license plates in images and video frames.

The project uses **Detectron2**, **RetinaNet**, **PyTorch**, and **OpenCV** to fine-tune a pretrained object detection model on a custom vehicle registration plate dataset.

---

## Project Goals

- Prepare a custom object detection dataset
- Register the dataset in Detectron2 format
- Visualize ground-truth license plate annotations
- Fine-tune a RetinaNet detector
- Run inference on validation images
- Evaluate the model using COCO-style metrics
- Apply the trained detector to video frames

> Target class: `vehicle-registration-plate`

## 🧰 Environment Setup

This section installs and imports the libraries required for training and inference.

> Note: Detectron2 installation can depend on the Python, PyTorch, and CUDA versions. If the install command fails, check the official Detectron2 installation guide for your environment.

In [ ]:
# Install dependencies
# Note: Detectron2 installation can vary depending on Python, PyTorch, and CUDA versions.
!pip install pycocotools
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

import os
import glob
import cv2
import random
import requests
import numpy as np
import matplotlib.pyplot as plt

from zipfile import ZipFile

import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import DatasetCatalog, MetadataCatalog, build_detection_test_loader
from detectron2.structures import BoxMode
from detectron2.evaluation import COCOEvaluator, inference_on_dataset

## 📦 Dataset Download

The dataset contains vehicle images and license plate bounding box annotations.

Expected structure after extraction:

```text
Dataset/
├── train/
│   └── Vehicle registration plate/
│       └── Label/
└── validation/
    └── Vehicle registration plate/
        └── Label/
```

In [ ]:
def download_and_unzip(url, save_path):
    print("Downloading and extracting dataset...", end=" ")
    response = requests.get(url)
    response.raise_for_status()

    with open(save_path, "wb") as file:
        file.write(response.content)

    try:
        with ZipFile(save_path) as zip_file:
            zip_file.extractall(os.path.split(save_path)[0])
        print("Done")
    except Exception as error:
        print(f"Failed to extract dataset: {error}")


DATASET_URL = "https://www.dropbox.com/s/k81ljpmzy3fgtx9/Dataset.zip?dl=1"

dataset_name = "Dataset"
dataset_zip_path = os.path.join(os.getcwd(), f"{dataset_name}.zip")
dataset_path = os.path.join(os.getcwd(), dataset_name)

if not os.path.exists(dataset_path):
    download_and_unzip(DATASET_URL, dataset_zip_path)
    os.remove(dataset_zip_path)

print("Dataset path:", dataset_path)

## 🗂️ Dataset Registration

Detectron2 expects datasets to be registered in a specific dictionary format.  
The function below parses the image files and annotation files, then registers both the training and validation splits.

In [ ]:
# Dataset configuration
data_root = os.path.join(os.getcwd(), "Dataset")

train_data_name = "vrp_train"
val_data_name = "vrp_val"
thing_classes = ["vehicle-registration-plate"]


def get_vrp_dicts(data_root, split):
    """Convert the license plate dataset into Detectron2 dictionary format."""

    img_dir = os.path.join(data_root, split, "Vehicle registration plate")

    # Some versions of the dataset use "Label"; others may use "Labels".
    label_dir = os.path.join(img_dir, "Label")
    if not os.path.isdir(label_dir):
        label_dir = os.path.join(img_dir, "Labels")

    image_paths = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        image_paths.extend(glob.glob(os.path.join(img_dir, ext)))
    image_paths = sorted(image_paths)

    dataset_dicts = []

    for image_path in image_paths:
        image = cv2.imread(image_path)
        if image is None:
            continue

        height, width = image.shape[:2]
        image_id = os.path.splitext(os.path.basename(image_path))[0]
        ann_path = os.path.join(label_dir, f"{image_id}.txt")

        record = {
            "file_name": image_path,
            "image_id": image_id,
            "height": height,
            "width": width,
        }

        objects = []

        if os.path.exists(ann_path):
            with open(ann_path, "r") as file:
                for line in file:
                    row = line.strip().split()
                    if len(row) < 5:
                        continue

                    try:
                        xmin, ymin, xmax, ymax = map(float, row[-4:])
                    except ValueError:
                        continue

                    # Keep bounding boxes inside image boundaries.
                    xmin = max(0.0, min(xmin, width - 1))
                    ymin = max(0.0, min(ymin, height - 1))
                    xmax = max(0.0, min(xmax, width - 1))
                    ymax = max(0.0, min(ymax, height - 1))

                    if xmax <= xmin or ymax <= ymin:
                        continue

                    objects.append({
                        "bbox": [xmin, ymin, xmax, ymax],
                        "bbox_mode": BoxMode.XYXY_ABS,
                        "category_id": 0,
                        "iscrowd": 0,
                    })

        record["annotations"] = objects
        dataset_dicts.append(record)

    return dataset_dicts


# Clear existing registrations to avoid duplicate dataset errors in notebooks.
for name in [train_data_name, val_data_name]:
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)
        MetadataCatalog.remove(name)

DatasetCatalog.register(train_data_name, lambda: get_vrp_dicts(data_root, "train"))
MetadataCatalog.get(train_data_name).set(thing_classes=thing_classes)

DatasetCatalog.register(val_data_name, lambda: get_vrp_dicts(data_root, "validation"))
MetadataCatalog.get(val_data_name).set(thing_classes=thing_classes)

print("Training images:", len(DatasetCatalog.get(train_data_name)))
print("Validation images:", len(DatasetCatalog.get(val_data_name)))

## 🖼️ Ground-Truth Visualization

Before training, it is useful to inspect a few validation samples and confirm that the bounding boxes are aligned correctly with the license plates.

In [ ]:
val_metadata = MetadataCatalog.get(val_data_name)
val_data_dicts = DatasetCatalog.get(val_data_name)

for sample in random.sample(val_data_dicts, min(3, len(val_data_dicts))):
    image = cv2.imread(sample["file_name"])

    visualizer = Visualizer(
        image[:, :, ::-1],
        metadata=val_metadata,
        scale=0.7
    )

    visualization = visualizer.draw_dataset_dict(sample)

    plt.figure(figsize=(12, 8))
    plt.imshow(visualization.get_image())
    plt.axis("off")
    plt.show()

## 🧠 Model Training

This project fine-tunes **RetinaNet with a ResNet-50 FPN backbone** using COCO-pretrained weights.

The model is configured for a single custom class: `vehicle-registration-plate`.

In [ ]:
# Detectron2 model configuration
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/retinanet_R_50_FPN_3x.yaml"))

# Use training data for training and validation data for evaluation.
cfg.DATASETS.TRAIN = (train_data_name,)
cfg.DATASETS.TEST = (val_data_name,)

cfg.DATALOADER.NUM_WORKERS = 4

# Start from COCO-pretrained RetinaNet weights.
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/retinanet_R_50_FPN_3x.yaml")

# Solver configuration
cfg.SOLVER.IMS_PER_BATCH = 4
cfg.SOLVER.BASE_LR = 0.0005
cfg.SOLVER.MAX_ITER = 2000

# The dataset has one custom class: vehicle registration plate.
cfg.MODEL.RETINANET.NUM_CLASSES = len(thing_classes)

# Output directory
cfg.OUTPUT_DIR = os.path.join(os.getcwd(), "outputs", "vrp")
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

print("Number of classes:", len(thing_classes))
print("Output directory:", cfg.OUTPUT_DIR)

In [ ]:
trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

## 🔍 Image Inference

After training, the final model weights are loaded and used to predict license plates on validation images.

In [ ]:
# Load the trained model for inference.
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.RETINANET.SCORE_THRESH_TEST = 0.5
cfg.DATASETS.TEST = (val_data_name,)

predictor = DefaultPredictor(cfg)

In [ ]:
test_data_dicts = DatasetCatalog.get(val_data_name)
test_metadata = MetadataCatalog.get(val_data_name)

for sample in random.sample(test_data_dicts, min(3, len(test_data_dicts))):
    print("Image:", sample["file_name"])

    image = cv2.imread(sample["file_name"])
    outputs = predictor(image)

    visualizer = Visualizer(
        image[:, :, ::-1],
        metadata=test_metadata,
        scale=0.8
    )

    prediction = visualizer.draw_instance_predictions(outputs["instances"].to("cpu"))

    plt.figure(figsize=(12, 8))
    plt.imshow(prediction.get_image())
    plt.axis("off")
    plt.show()

## 📊 COCO-Style Evaluation

The trained detector is evaluated using COCO-style object detection metrics, including:

- **AP**: Average Precision across multiple IoU thresholds
- **AP50**: Average Precision at IoU = 0.50
- **AP75**: Average Precision at IoU = 0.75

These metrics help measure how accurately the model localizes license plates.

In [ ]:
eval_dir = os.path.join(cfg.OUTPUT_DIR, "coco_eval")
os.makedirs(eval_dir, exist_ok=True)

evaluator = COCOEvaluator(
    dataset_name=val_data_name,
    tasks=("bbox",),
    distributed=False,
    output_dir=eval_dir
)

val_loader = build_detection_test_loader(cfg, val_data_name)

evaluation_results = inference_on_dataset(trainer.model, val_loader, evaluator)
evaluation_results

## 🎥 Video Inference

The trained detector can also be applied frame by frame to a video file.

This is useful for demonstrating the project as a practical computer vision pipeline, not just a notebook experiment.

In [ ]:
def run_video_inference(video_path, output_dir="anpd_out"):
    """Run license plate detection on a video file and save the annotated output."""

    video = cv2.VideoCapture(video_path)

    if not video.isOpened():
        print("Error: could not open video file.")
        return None

    os.makedirs(output_dir, exist_ok=True)

    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fps = video.get(cv2.CAP_PROP_FPS)
    if fps == 0 or fps != fps:
        fps = 25.0

    base_name = os.path.splitext(os.path.basename(video_path))[0]
    output_video_path = os.path.join(output_dir, f"{base_name}_detected.avi")

    writer = cv2.VideoWriter(
        output_video_path,
        cv2.VideoWriter_fourcc(*"XVID"),
        fps,
        (width, height)
    )

    if not writer.isOpened():
        print("Error: VideoWriter failed to open.")
        return None

    frame_index = 0

    while True:
        ret, frame = video.read()
        if not ret:
            break

        outputs = predictor(frame)

        visualizer = Visualizer(
            frame[:, :, ::-1],
            metadata=test_metadata,
            scale=0.8
        )

        visualization = visualizer.draw_instance_predictions(outputs["instances"].to("cpu"))
        result_frame = visualization.get_image()[:, :, ::-1]

        if result_frame.shape[1] != width or result_frame.shape[0] != height:
            result_frame = cv2.resize(result_frame, (width, height))

        writer.write(result_frame)

        # Save a few frames as assets for the GitHub README.
        if frame_index in [0, 30, 60]:
            cv2.imwrite(os.path.join(output_dir, f"sample_frame_{frame_index:03d}.jpg"), result_frame)

        frame_index += 1

    video.release()
    writer.release()

    print("Saved output video:", output_video_path)
    print("Frames processed:", frame_index)

    return output_video_path


# Example:
# run_video_inference("project3-input-video.mp4")

## ▶️ Demo Video

If you upload your output video to YouTube, replace the video ID below with your own video ID.

In [ ]:
from IPython.display import YouTubeVideo, Markdown, display

# Replace this ID with your own demo video ID if you upload the result to YouTube.
demo_video_id = "zeSJZR8pQb8"

display(YouTubeVideo(demo_video_id, width=720, height=405))
display(Markdown(f"[▶ Watch demo video on YouTube](https://www.youtube.com/watch?v={demo_video_id})"))

## ✅ Next Improvements

This project can be extended in several professional directions:

- Add OCR to read the detected license plate text
- Convert notebook code into reusable Python scripts
- Add FastAPI deployment for image upload and inference
- Add Docker support
- Add MLflow or TensorBoard experiment tracking
- Improve the dataset with more lighting conditions, angles, and plate styles
- Export the model for optimized inference
- Add real-time webcam inference

---

## Portfolio Summary

This notebook demonstrates a complete object detection workflow: dataset preparation, model configuration, transfer learning, inference, evaluation, and video processing.

It is suitable as a GitHub portfolio project for computer vision, machine learning, and AI engineering roles.